In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Cross-City Crash Data: Exploratory Data Analysis\n",
    "\n",
    "**Status:** scaffold. Analysis cells are not yet filled in — they\n",
    "require the curated Parquet output from Member B's transformation\n",
    "layer, which is not yet built. Each section below states what the\n",
    "analysis *will* show once the curated layer exists.\n",
    "\n",
    "**Data source:** the curated layer (`data/curated/`, partitioned by\n",
    "`source_id/year/month`), not the raw layer. Reading the raw layer\n",
    "would re-implement the transformation logic that the pipeline exists\n",
    "to centralize.\n",
    "\n",
    "## Rule for this notebook: structural nulls are not findings\n",
    "\n",
    "Several canonical fields are **structurally null for some sources**\n",
    "because the source does not collect them — this is documented in\n",
    "`docs/data_dictionary.md` (section \"Cross-Source Gaps at a Glance\") and\n",
    "in `docs/data_contract.md` (section 4). Any cross-source comparison\n",
    "must account for this, or it will read a data gap as a finding.\n",
    "\n",
    "Concretely: a naive chart of **hit-and-run rate by city** shows 0% for\n",
    "NYC and UK. That is not evidence NYC and the UK have no hit-and-runs —\n",
    "it is evidence the source does not record the field. The same applies\n",
    "to `crash_type` (Chicago only), `primary_cause` (Chicago + NYC only),\n",
    "`weather_condition` / `lighting_condition` / `posted_speed_limit_mph`\n",
    "(Chicago + UK only), `num_killed` (Chicago + NYC only), and the\n",
    "`severity = 'serious'` bucket (Chicago + UK only).\n",
    "\n",
    "Every cross-source chart in this notebook either (a) filters to the\n",
    "sources that actually populate the field, or (b) annotates the nulls\n",
    "as structural in the caption."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 0. Setup\n",
    "\n",
    "Imports and a single helper for loading one partition of curated\n",
    "Parquet. Reading a single partition (`source_id` + year) rather than\n",
    "the full dataset is the access pattern the pipeline is designed for —\n",
    "it also exercises the partition-pruning requirement from the project\n",
    "proposal (section 9)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pathlib import Path\n",
    "\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "plt.rcParams[\"figure.dpi\"] = 110\n",
    "\n",
    "CURATED_ROOT = Path(\"../data/curated\")\n",
    "\n",
    "def load_source(source_id: str, year: int | None = None) -> pd.DataFrame:\n",
    "    \"\"\"Load curated Parquet for one source, optionally one year.\n",
    "\n",
    "    Partition scheme: data/curated/source_id=<id>/year=<YYYY>/month=<MM>/\n",
    "    Reading one partition (rather than the full tree) is the pipeline's\n",
    "    intended access pattern and is what the live demo exercises.\n",
    "    \"\"\"\n",
    "    base = CURATED_ROOT / f\"source_id={source_id}\"\n",
    "    if year is not None:\n",
    "        base = base / f\"year={year}\"\n",
    "    if not base.exists():\n",
    "        raise FileNotFoundError(\n",
    "            f\"Curated partition not found: {base}. Member B's transformation \"\n",
    "            f\"layer must be run first; see TASKS.md.\"\n",
    "        )\n",
    "    return pd.read_parquet(base)\n",
    "\n",
    "SOURCES = [\"chicago_us\", \"nyc_us\", \"uk_stats19\"]"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Volume and coverage\n",
    "\n",
    "Rows per source, per year. Purpose: confirm each source's date range\n",
    "matches what profiling found (Chicago ~2013–present, UK 2021–2025,\n",
    "NYC last 5 years). Any source with an unexpected shape here is worth\n",
    "investigating before trusting later charts."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# df = pd.concat([load_source(s) for s in SOURCES], ignore_index=True)\n",
    "# df.groupby([\"source_id\", df[\"crash_date_key\"].dt.year]).size().unstack(0)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Temporal patterns\n",
    "\n",
    "### 2.1 Hour of day\n",
    "\n",
    "Crashes per hour of day, per source. Hours derived from\n",
    "`crash_timestamp_utc` converted to the source's local timezone\n",
    "(`source_local_timezone` on `dim_source`) — comparing UTC hours\n",
    "across cities would smear the morning and evening peaks into different\n",
    "clock positions for each city."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Convert to local time per source before plotting.\n",
    "# df[\"local_hour\"] = df.apply(\n",
    "#     lambda r: r[\"crash_timestamp_utc\"].tz_convert(TZ[r[\"source_id\"]]).hour, axis=1\n",
    "# )\n",
    "# sns.histplot(data=df, x=\"local_hour\", hue=\"source_id\", multiple=\"dodge\", bins=24)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 2.2 Day of week\n",
    "\n",
    "Crashes per weekday, per source. Uses `dim_date.day_of_week` via\n",
    "`crash_date_key` rather than re-deriving from the timestamp — one\n",
    "definition of \"weekday\" across the notebook."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Join dim_date or use df[\"crash_date_key\"].dt.dayofweek.\n",
    "# sns.countplot(data=df, x=\"day_of_week\", hue=\"source_id\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 2.3 Weekday vs weekend\n",
    "\n",
    "Crashes per weekday/weekend, per source. Uses the `is_weekend` flag\n",
    "on `dim_date` for the same reason as above."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# sns.countplot(data=df, x=\"is_weekend\", hue=\"source_id\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Severity\n",
    "\n",
    "### 3.1 Severity distribution, per source\n",
    "\n",
    "Counts of `fatal` / `serious` / `minor` / `none` / `unknown` per source.\n",
    "\n",
    "**Structural gap to annotate:** NYC cannot produce `serious` at all —\n",
    "the source has no categorical severity field, only counts. The bar\n",
    "for NYC's `serious` will be zero or absent by construction, not\n",
    "because NYC has no serious-injury crashes. This must be annotated in\n",
    "the chart caption."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# sns.countplot(data=df, x=\"severity\", hue=\"source_id\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 3.2 Severity share by source\n",
    "\n",
    "Normalized severity distribution (proportions, not counts), so\n",
    "sources with vastly different volumes (Chicago ~1.1M, UK ~514K,\n",
    "NYC ~5 yrs of a high-volume feed) can be compared on shape rather\n",
    "than absolute count. Still subject to the NYC `serious` gap above."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# pd.crosstab(df[\"source_id\"], df[\"severity\"], normalize=\"index\").plot.bar(stacked=True)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 3.3 Injury and fatality counts\n",
    "\n",
    "Distribution of `num_injured_total` and `num_killed` per source.\n",
    "\n",
    "**Structural gap to annotate:** `num_killed` is null for `uk_stats19`\n",
    "(STATS19 casualty detail lives in a separate table not ingested). The\n",
    "UK box/violin will be absent — annotate this. For NYC, `num_killed`\n",
    "comes from summing the pedestrian/cyclist/motorist components, not\n",
    "the (88% null) aggregate field — this is documented in `nyc.yaml`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# sns.boxplot(data=df.dropna(subset=[\"num_killed\"]), x=\"source_id\", y=\"num_killed\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Cross-source comparisons\n",
    "\n",
    "### 4.1 Fatal crash rate by source, coordinates-only subset\n",
    "\n",
    "Fatal-crash percentage per source, restricted to\n",
    "`has_valid_coordinates = TRUE`. Using the coordinates subset is\n",
    "necessary for any follow-on mapping or clustering, and keeps this\n",
    "number comparable to the spatial analysis later.\n",
    "\n",
    "**Caveat to state in the caption:** reporting thresholds and coverage\n",
    "differ by source, so this compares *rates within each source's own\n",
    "universe*, not a common population."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# df_valid = df[df[\"has_valid_coordinates\"]]\n",
    "# df_valid.groupby(\"source_id\")[\"severity\"].value_counts(normalize=True).unstack()[\"fatal\"]"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 4.2 Where cross-source comparison is NOT valid\n",
    "\n",
    "This cell is a checklist, not an analysis. It documents the fields a\n",
    "naive cross-source comparison would misinterpret, and the sources\n",
    "each one is restricted to. It exists so a reader of this notebook\n",
    "sees the constraint, not just a claim in the writeup.\n",
    "\n",
    "| Field | Valid sources | Invalid sources | Reason |\n",
    "|---|---|---|---|\n",
    "| `crash_type` | chicago_us | nyc_us, uk_stats19 | Neither source classifies crashes by physical type |\n",
    "| `primary_cause` | chicago_us, nyc_us | uk_stats19 | STATS19 contributory factors live in a separate table |\n",
    "| `weather_condition` | chicago_us, uk_stats19 | nyc_us | Not collected by NYC |\n",
    "| `lighting_condition` | chicago_us, uk_stats19 | nyc_us | Not collected by NYC |\n",
    "| `posted_speed_limit_mph` | chicago_us, uk_stats19 | nyc_us | Not collected by NYC |\n",
    "| `num_killed` | chicago_us, nyc_us | uk_stats19 | STATS19 casualty detail separate |\n",
    "| `hit_and_run` | chicago_us | nyc_us, uk_stats19 | Defaults to false elsewhere (no source signal) |\n",
    "| `severity = 'serious'` | chicago_us, uk_stats19 | nyc_us | NYC has no categorical severity field |\n",
    "| `police_notified_timestamp_utc` | chicago_us | nyc_us, uk_stats19 | No equivalent field in either source |\n",
    "| `city` | chicago_us, nyc_us | uk_stats19 | UK is a national (GB-wide) dataset |"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Data quality signals surfaced by the curated layer\n",
    "\n",
    "### 5.1 Coordinate coverage\n",
    "\n",
    "Percentage of rows with `has_valid_coordinates = TRUE`, per source.\n",
    "From profiling (documented in `docs/data_dictionary.md`): Chicago\n",
    "0.78% missing, NYC 2.3% sentinel `(0,0)`, UK ~0.01% missing. The point\n",
    "of this cell is to confirm that what the notebook sees matches what\n",
    "profiling found."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# df.groupby(\"source_id\")[\"has_valid_coordinates\"].mean()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 5.2 Data-quality outcomes from `dq_run_log`\n",
    "\n",
    "The `dq_run_log` table records which checks passed/failed per batch.\n",
    "This cell queries it to show whether the batches currently in the\n",
    "curated layer passed all checks.\n",
    "\n",
    "Requires a live Postgres connection to the curated database (see\n",
    "`.env` for credentials). Fill in once Member B's load stage is in\n",
    "place and there are real `dq_run_log` rows to report."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# import os, psycopg2\n",
    "# conn = psycopg2.connect(\n",
    "#     host=os.environ[\"PIPELINE_POSTGRES_HOST\"],\n",
    "#     port=os.environ[\"PIPELINE_POSTGRES_PORT\"],\n",
    "#     dbname=os.environ[\"PIPELINE_POSTGRES_DB\"],\n",
    "#     user=os.environ[\"PIPELINE_POSTGRES_USER\"],\n",
    "#     password=os.environ[\"PIPELINE_POSTGRES_PASSWORD\"],\n",
    "# )\n",
    "# pd.read_sql(\"SELECT source_id, check_name, status, rows_failed, run_timestamp \"\n",
    "#             \"FROM dq_run_log ORDER BY run_timestamp DESC LIMIT 50\", conn)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Notes for whoever fills this in\n",
    "\n",
    "1. **Do not read the raw layer.** The curated Parquet layer already has\n",
    "   the canonical schema, harmonized values, and cleared sentinels.\n",
    "   Reading `data/raw/` re-implements logic the pipeline exists to\n",
    "   centralize and will produce numbers that disagree with the\n",
    "   Postgres table.\n",
    "\n",
    "2. **Always convert to local time before plotting hourly patterns.**\n",
    "   `crash_timestamp_utc` is UTC; a UTC-only hour histogram puts each\n",
    "   city's morning peak at a different clock position and makes the\n",
    "   cross-city comparison meaningless. The source's local timezone is\n",
    "   on `dim_source.source_local_timezone`.\n",
    "\n",
    "3. **Annotate structural nulls, do not silently drop them.** If a\n",
    "   chart excludes UK rows because `num_killed` is null, the caption\n",
    "   should say so. A reader who does not know the gap will interpret\n",
    "   the absence of a bar as a zero.\n",
    "\n",
    "4. **Use `has_valid_coordinates` for any spatial filtering.** Do not\n",
    "   filter on `latitude IS NOT NULL AND longitude IS NOT NULL`\n",
    "   directly; the sentinel `(0,0)` rows would pass that check and place\n",
    "   crashes in the Gulf of Guinea.\n",
    "\n",
    "5. **Report per-source, not just aggregate.** A single count across\n",
    "   all sources hides the fact that they cover different geographies,\n",
    "   different legal definitions of severity, and different time spans."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}